# Phase 3 — Analyze
## 03 —SOA / Sell-Out Allowance Analysis

## Objective
To analyse Sell-Out Allowance (SOA) activity across products and promotional validity periods, and determine how supplier/customer allowance support relates to product sales, revenue, and commercial performance.

In [6]:
# ============================================================
# LOAD GOVERNED SOA DATA
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent.parent

SOA_PATH = PROJECT_ROOT / "data" / "processed" / "fact_soa.csv"
DIM_PRODUCT_PATH = PROJECT_ROOT / "data" / "processed" / "dim_product.csv"
SALES_PATH = PROJECT_ROOT / "data" / "processed" / "fact_sales.csv"

fact_soa = pd.read_csv(SOA_PATH)
dim_product = pd.read_csv(DIM_PRODUCT_PATH)

print("GOVERNED SOA DATA")
print("=" * 80)

print(f"Rows    : {len(fact_soa):,}")
print(f"Columns : {fact_soa.shape[1]:,}")

print("\nCOLUMNS")
print("=" * 80)

for col in fact_soa.columns:
    print(col)

print("\nSAMPLE")
display(fact_soa.head(10))

GOVERNED SOA DATA
Rows    : 412
Columns : 11

COLUMNS
SOA_Record_ID
Product_ID
Product_Key
Model
Description
Starts
Ends
Window_Days
SOA
Original_Ends
Date_Correction_Flag

SAMPLE


,SOA_Record_ID,Product_ID,Product_Key,Model,Description,Starts,Ends,Window_Days,SOA,Original_Ends,Date_Correction_Flag
0,1,11,107833-01,107833-01,Dyson Supersonic Hair Dryer,2025-12-31,2026-02-03,35,66.50,2026-02-03,False
1,2,36,161818-01,161818-01,Dyson Supersonic Ceramic,2025-12-31,2026-02-03,35,66.50,2026-02-03,False
2,3,39,19750,19750,Russell Hobbs Rice Cooker 1.8Ltr,2026-02-01,2026-02-28,28,5.00,2026-02-28,False
3,4,48,21270,21270,Russell Hobbs White Textures Jug,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
4,5,49,21271,21271,Russell Hobbs Black Textures Jug,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
5,6,50,21274,21274,Russell Hobbs Texture Kettle,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
6,7,52,21640,21640,Russell Hobbs White Textures 2,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
7,8,53,21641,21641,RUSSELL HOBBS BLACK,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
8,9,54,21644,21644,Russell Hobbs Textures Toaster,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
9,10,57,23211,23211,Luna Moonlight Grey Quiet Boil,2026-02-01,2026-02-28,28,17.08,2026-02-28,False


### Step 1 — SOA Population & Integrity

In [2]:
soa = fact_soa.copy()

# ------------------------------------------------------------
# 1. Ensure analytical datatypes
# ------------------------------------------------------------

soa["Starts"] = pd.to_datetime(soa["Starts"], errors="coerce")
soa["Ends"] = pd.to_datetime(soa["Ends"], errors="coerce")
soa["SOA"] = pd.to_numeric(soa["SOA"], errors="coerce")
soa["Window_Days"] = pd.to_numeric(soa["Window_Days"], errors="coerce")

# ------------------------------------------------------------
# 2. Required-field checks
# ------------------------------------------------------------

required_columns = [
    "SOA_Record_ID",
    "Product_ID",
    "Product_Key",
    "Model",
    "Starts",
    "Ends",
    "Window_Days",
    "SOA"
]

missing_columns = [
    col for col in required_columns
    if col not in soa.columns
]

duplicate_records = soa["SOA_Record_ID"].duplicated().sum()

missing_critical = soa[
    ["SOA_Record_ID", "Product_ID", "Starts", "Ends", "SOA"]
].isna().sum().sum()

invalid_windows = (soa["Ends"] < soa["Starts"]).sum()

negative_soa = (soa["SOA"] < 0).sum()

# ------------------------------------------------------------
# 3. Population summary
# ------------------------------------------------------------

print("SOA POPULATION & INTEGRITY")
print("=" * 80)

print(f"SOA records             : {len(soa):,}")
print(f"Unique products         : {soa['Product_ID'].nunique():,}")
print(f"Unique models           : {soa['Model'].nunique():,}")
print()

print("DATA INTEGRITY")
print("=" * 80)

print(f"Missing required columns: {missing_columns}")
print(f"Duplicate SOA_Record_ID : {duplicate_records:,}")
print(f"Missing critical values : {missing_critical:,}")
print(f"Invalid date windows    : {invalid_windows:,}")
print(f"Negative SOA values     : {negative_soa:,}")

print()
print(
    "Population valid       :",
    len(missing_columns) == 0
    and duplicate_records == 0
    and missing_critical == 0
    and invalid_windows == 0
)

SOA POPULATION & INTEGRITY
SOA records             : 412
Unique products         : 412
Unique models           : 412

DATA INTEGRITY
Missing required columns: []
Duplicate SOA_Record_ID : 0
Missing critical values : 0
Invalid date windows    : 0
Negative SOA values     : 0

Population valid       : True


### Step 2 — SOA Allowance & Validity Window Analysis

In [3]:
print("SOA ALLOWANCE ANALYSIS")
print("=" * 80)

print(f"Products with SOA       : {len(soa):,}")
print(f"Total SOA value         : £{soa['SOA'].sum():,.2f}")
print(f"Average SOA per product : £{soa['SOA'].mean():,.2f}")
print(f"Median SOA              : £{soa['SOA'].median():,.2f}")
print(f"Minimum SOA             : £{soa['SOA'].min():,.2f}")
print(f"Maximum SOA             : £{soa['SOA'].max():,.2f}")

print()
print("SOA DISTRIBUTION")
print("=" * 80)

print(
    soa["SOA"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
    .round(2)
)

print()
print("VALIDITY WINDOW ANALYSIS")
print("=" * 80)

print(f"Average window          : {soa['Window_Days'].mean():.2f} days")
print(f"Median window           : {soa['Window_Days'].median():.2f} days")
print(f"Shortest window         : {soa['Window_Days'].min():.0f} days")
print(f"Longest window          : {soa['Window_Days'].max():.0f} days")

print()
print("DATE GOVERNANCE")
print("=" * 80)

if "Date_Correction_Flag" in soa.columns:
    print(soa["Date_Correction_Flag"].value_counts(dropna=False))

SOA ALLOWANCE ANALYSIS
Products with SOA       : 412
Total SOA value         : £29,704.87
Average SOA per product : £72.10
Median SOA              : £29.75
Minimum SOA             : £0.43
Maximum SOA             : £918.48

SOA DISTRIBUTION
count    412.00
mean      72.10
std      119.58
min        0.43
25%       10.10
50%       29.75
75%       66.64
90%      190.58
95%      315.62
max      918.48
Name: SOA, dtype: float64

VALIDITY WINDOW ANALYSIS
Average window          : 23.97 days
Median window           : 25.00 days
Shortest window         : 5 days
Longest window          : 63 days

DATE GOVERNANCE
Date_Correction_Flag
False    411
True       1
Name: count, dtype: int64


### Step 3 — SOA × Sales Integration

In [7]:
# Load governed sales fact
fact_sales = pd.read_csv(SALES_PATH)

# Standardise Product_ID type
soa["Product_ID"] = pd.to_numeric(
    soa["Product_ID"], errors="coerce"
).astype("Int64")

fact_sales["Product_ID"] = pd.to_numeric(
    fact_sales["Product_ID"], errors="coerce"
).astype("Int64")


# ------------------------------------------------------------
# Product populations
# ------------------------------------------------------------

soa_product_ids = set(soa["Product_ID"].dropna().unique())
sales_product_ids = set(fact_sales["Product_ID"].dropna().unique())

matched_ids = soa_product_ids & sales_product_ids
soa_without_sales = soa_product_ids - sales_product_ids


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("SOA × SALES PRODUCT BRIDGE")
print("=" * 80)

print(f"SOA products              : {len(soa_product_ids):,}")
print(f"Sales products            : {len(sales_product_ids):,}")
print(f"SOA products with sales   : {len(matched_ids):,}")
print(f"SOA products without sales: {len(soa_without_sales):,}")

print()

match_rate = (
    len(matched_ids) / len(soa_product_ids) * 100
    if len(soa_product_ids) > 0
    else 0
)

print(f"SOA → Sales match rate    : {match_rate:.2f}%")

print()
print("BRIDGE VALIDATION")
print("=" * 80)

print(
    "All SOA Product_IDs valid:",
    soa["Product_ID"].notna().all()
)

print(
    "Matched + unmatched reconcile:",
    len(matched_ids) + len(soa_without_sales) == len(soa_product_ids)
)

SOA × SALES PRODUCT BRIDGE
SOA products              : 412
Sales products            : 767
SOA products with sales   : 62
SOA products without sales: 350

SOA → Sales match rate    : 15.05%

BRIDGE VALIDATION
All SOA Product_IDs valid: True
Matched + unmatched reconcile: True


### Step 4 — SOA × Sales Month Overlap Analysis

In [10]:
# Map Source_Month to actual reporting periods
month_map = {
    "Nov": "2025-11",
    "Dec": "2025-12",
    "Jan": "2026-01"
}

fact_sales["Sales_Month"] = pd.to_datetime(
    fact_sales["Source_Month"].map(month_map),
    format="%Y-%m"
).dt.to_period("M")

# Ensure SOA dates are datetime
soa["Starts"] = pd.to_datetime(soa["Starts"])
soa["Ends"] = pd.to_datetime(soa["Ends"])

# SOA months covered by each allowance window
soa["SOA_Start_Month"] = soa["Starts"].dt.to_period("M")
soa["SOA_End_Month"] = soa["Ends"].dt.to_period("M")

# Join only products appearing in both datasets
soa_sales = fact_sales.merge(
    soa[
        [
            "Product_ID",
            "SOA_Record_ID",
            "SOA",
            "Starts",
            "Ends",
            "SOA_Start_Month",
            "SOA_End_Month"
        ]
    ],
    on="Product_ID",
    how="inner"
)

# Does the sales reporting month overlap the SOA window?
soa_sales["SOA_Month_Overlap"] = (
    (soa_sales["Sales_Month"] >= soa_sales["SOA_Start_Month"]) &
    (soa_sales["Sales_Month"] <= soa_sales["SOA_End_Month"])
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

overlap = soa_sales[soa_sales["SOA_Month_Overlap"]].copy()

print("SOA × SALES MONTH OVERLAP")
print("=" * 80)

print(f"SOA products with sales      : {soa_sales['Product_ID'].nunique():,}")
print(f"Matched sales records        : {len(soa_sales):,}")
print(f"Records in SOA-overlap months: {len(overlap):,}")
print(f"Products with month overlap  : {overlap['Product_ID'].nunique():,}")

print()
print("SALES ACTIVITY IN OVERLAP MONTHS")
print("=" * 80)

print(f"Net units    : {overlap['Sold Period'].sum():,.0f}")
print(f"Sales value  : £{overlap['Sales Value'].sum():,.2f}")
print(f"Gross profit : £{overlap['Profit'].sum():,.2f}")

print()
print("VALIDATION")
print("=" * 80)

print(
    "Overlap records valid:",
    overlap["SOA_Month_Overlap"].all()
)

print(
    "Product population valid:",
    soa_sales["Product_ID"].nunique() == 62
)

SOA × SALES MONTH OVERLAP
SOA products with sales      : 62
Matched sales records        : 88
Records in SOA-overlap months: 8
Products with month overlap  : 8

SALES ACTIVITY IN OVERLAP MONTHS
Net units    : 9
Sales value  : £2,430.83
Gross profit : £-449.36

VALIDATION
Overlap records valid: True
Product population valid: True


### Step 5 — SOA Commercial Exposure Analysis

In [11]:
soa_exposure = overlap.copy()

# Potential allowance exposure
soa_exposure["Potential_SOA_Exposure"] = (
    soa_exposure["Sold Period"] * soa_exposure["SOA"]
)

# Indicative profit after SOA support
soa_exposure["Indicative_Profit_After_SOA"] = (
    soa_exposure["Profit"] + soa_exposure["Potential_SOA_Exposure"]
)

# Profitability status
soa_exposure["Pre_SOA_Loss"] = soa_exposure["Profit"] < 0

soa_exposure["Indicative_Post_SOA_Profitable"] = (
    soa_exposure["Indicative_Profit_After_SOA"] >= 0
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("SOA COMMERCIAL EXPOSURE")
print("=" * 80)

print(f"Products analysed             : {soa_exposure['Product_ID'].nunique():,}")
print(f"Net units                     : {soa_exposure['Sold Period'].sum():,.0f}")
print(f"Sales value                   : £{soa_exposure['Sales Value'].sum():,.2f}")
print(f"Gross profit before SOA       : £{soa_exposure['Profit'].sum():,.2f}")
print(
    f"Potential SOA exposure        : "
    f"£{soa_exposure['Potential_SOA_Exposure'].sum():,.2f}"
)
print(
    f"Indicative profit after SOA   : "
    f"£{soa_exposure['Indicative_Profit_After_SOA'].sum():,.2f}"
)

print()
print("PROFITABILITY IMPACT")
print("=" * 80)

print(
    f"Loss-making records before SOA: "
    f"{soa_exposure['Pre_SOA_Loss'].sum():,}"
)

print(
    f"Indicatively profitable after : "
    f"{soa_exposure['Indicative_Post_SOA_Profitable'].sum():,}"
)

print()
print("VALIDATION")
print("=" * 80)

print(
    "Exposure calculation valid:",
    (
        soa_exposure["Potential_SOA_Exposure"]
        ==
        soa_exposure["Sold Period"] * soa_exposure["SOA"]
    ).all()
)

print(
    "Population reconciles:",
    soa_exposure["Product_ID"].nunique()
    == overlap["Product_ID"].nunique()
)

SOA COMMERCIAL EXPOSURE
Products analysed             : 8
Net units                     : 9
Sales value                   : £2,430.83
Gross profit before SOA       : £-449.36
Potential SOA exposure        : £562.25
Indicative profit after SOA   : £112.89

PROFITABILITY IMPACT
Loss-making records before SOA: 7
Indicatively profitable after : 5

VALIDATION
Exposure calculation valid: True
Population reconciles: True


### Step 6 — Final SOA Analysis Gate

In [12]:

# Core gate conditions
population_valid = (
    len(soa) == 412
    and soa["Product_ID"].nunique() == 412
)

bridge_valid = (
    len(matched_ids) + len(soa_without_sales)
    == len(soa_product_ids)
)

overlap_valid = (
    overlap["SOA_Month_Overlap"].all()
    and overlap["Product_ID"].nunique() == 8
)

exposure_valid = (
    (
        soa_exposure["Potential_SOA_Exposure"]
        ==
        soa_exposure["Sold Period"] * soa_exposure["SOA"]
    ).all()
)

final_gate = all([
    population_valid,
    bridge_valid,
    overlap_valid,
    exposure_valid
])

print("FINAL SOA ANALYSIS GATE")
print("=" * 80)

print(f"SOA products                : {soa['Product_ID'].nunique():,}")
print(f"SOA products with sales     : {len(matched_ids):,}")
print(f"Products with month overlap : {overlap['Product_ID'].nunique():,}")
print(f"Overlap net units           : {overlap['Sold Period'].sum():,.0f}")

print()
print(f"Gross profit before SOA     : £{soa_exposure['Profit'].sum():,.2f}")
print(
    f"Potential SOA exposure      : "
    f"£{soa_exposure['Potential_SOA_Exposure'].sum():,.2f}"
)
print(
    f"Indicative profit after SOA : "
    f"£{soa_exposure['Indicative_Profit_After_SOA'].sum():,.2f}"
)

print()
print("GATE CONDITIONS")
print("=" * 80)

print(f"SOA population integrity    : {population_valid}")
print(f"SOA-sales bridge integrity  : {bridge_valid}")
print(f"Month-overlap integrity     : {overlap_valid}")
print(f"SOA exposure integrity      : {exposure_valid}")

print()
print("FINAL SOA GATE")
print("=" * 80)
print("STATUS:", "PASS" if final_gate else "REVIEW")

FINAL SOA ANALYSIS GATE
SOA products                : 412
SOA products with sales     : 62
Products with month overlap : 8
Overlap net units           : 9

Gross profit before SOA     : £-449.36
Potential SOA exposure      : £562.25
Indicative profit after SOA : £112.89

GATE CONDITIONS
SOA population integrity    : True
SOA-sales bridge integrity  : True
Month-overlap integrity     : True
SOA exposure integrity      : True

FINAL SOA GATE
STATUS: PASS


# SOA / Sell-Out Allowance Analysis — Key Findings

## SOA Portfolio

- The governed SOA dataset contains **412 products**, each with one valid SOA record.
- Total listed SOA value across the portfolio is **£29,704.87**.
- Average SOA is **£72.10**, while the median is **£29.75**, indicating a right-skewed allowance distribution with a smaller number of high-value allowances.
- SOA validity windows have a median duration of **25 days** and range from **5 to 63 days**.
- One governed date correction is present in the dataset.

## SOA × Sales Coverage

- Of the **412 SOA products**, **62** have sales history in the available Nov 2025–Jan 2026 sales dataset.
- This represents a **15.05% SOA-to-sales product overlap**.
- Only **8 products** have sales reporting months that overlap their SOA validity periods.
- These overlapping records account for **9 net units** and **£2,430.83 in sales value**.

## Commercial Impact

- The overlapping population generated **-£449.36 gross profit before SOA support**.
- Estimated potential SOA exposure is **£562.25**.
- Applying this potential support produces an **indicative gross profit of £112.89**.
- **7 of the 8 overlapping records were loss-making before SOA**, while **5 records are indicatively profitable after SOA support**.
- This suggests that SOA can materially improve product-level margin and may recover profitability on some otherwise loss-making sales.

## Analytical Limitation

- Sales data is available only at **monthly reporting level** (`Nov`, `Dec`, `Jan`), whereas SOA validity is defined using exact start and end dates.
- Therefore, the analysis identifies **sales-month overlap with SOA periods**, not confirmed transaction-level SOA eligibility.
- The calculated **£562.25 is potential SOA exposure**, not confirmed allowance earned or claimed.
- Transaction-level sales dates would be required for exact SOA attribution and realised margin measurement.

## Conclusion

The analysis demonstrates that SOA support has the potential to materially affect retail product profitability. Within the observable overlap population, potential SOA support is sufficient to move aggregate gross profit from **-£449.36 to an indicative £112.89**.

The result should be treated as directional rather than confirmed because of the monthly granularity of the available sales data.